# 12 Evidence Synthesis Engine

**Project:** Insurance Fraud Detection Assistant

**Notebook:** `12-evidence-synthesis-engine.ipynb`

In [1]:
# Start coding here

In [3]:
# ==========================================
# Notebook 12
# Evidence Synthesis Engine
# ==========================================

import pandas as pd
import numpy as np
import json

from transformers import pipeline

In [4]:
cross_encoder_df = pd.read_csv("../data/cross_encoder_verification.csv")

template_fraud_df = pd.read_csv("../data/template_fraud_candidates.csv")

ring_candidates_df = pd.read_csv("../data/semantic_ring_candidates.csv")

claims_df = pd.read_csv("../data/insurance_claims.csv")

In [5]:
print("Claims:", len(claims_df))

print("Cross Encoder Results:", len(cross_encoder_df))

print("Template Fraud Candidates:", len(template_fraud_df))

print("Ring Candidates:", len(ring_candidates_df))

Claims: 15
Cross Encoder Results: 16
Template Fraud Candidates: 11
Ring Candidates: 9


In [6]:
llm = pipeline("text-generation", model="google/flan-t5-base")

Device set to use cpu
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalLM', 'Exaone4ForCausalLM', 'FalconForCausalLM', 'FalconH1ForCausalLM', 'FalconMambaForCausalLM', 'FlexOlmoF

In [7]:
def collect_claim_evidence(claim_id):

    evidence = {}

    claim_record = claims_df[claims_df["claim_id"] == claim_id]

    if len(claim_record) == 0:

        return None

    claim_record = claim_record.iloc[0]

    evidence["claim_id"] = claim_id

    evidence["claimant_statement"] = claim_record["claimant_statement"]

    evidence["police_report"] = claim_record["police_report"]

    evidence["adjuster_notes"] = claim_record["adjuster_notes"]

    cross_match = cross_encoder_df[cross_encoder_df["claim_id"] == claim_id]

    if len(cross_match) > 0:

        evidence["verification_score"] = float(cross_match.iloc[0]["normalized_score"])

    else:

        evidence["verification_score"] = None

    template_matches = template_fraud_df[
        (template_fraud_df["claim_1"] == claim_id)
        | (template_fraud_df["claim_2"] == claim_id)
    ]

    evidence["template_match_count"] = len(template_matches)

    ring_matches = ring_candidates_df[
        (ring_candidates_df["claim_1"] == claim_id)
        | (ring_candidates_df["claim_2"] == claim_id)
    ]

    evidence["ring_match_count"] = len(ring_matches)

    return evidence

In [8]:
collect_claim_evidence("CLM001")

{'claim_id': 'CLM001',
 'claimant_statement': 'A vehicle rear-ended me while I was waiting at a traffic signal.',
 'police_report': 'Witnesses confirmed another driver caused the accident.',
 'adjuster_notes': 'Section international though many movement.',
 'verification_score': 0.1821472668914981,
 'template_match_count': 1,
 'ring_match_count': 2}

In [9]:
def calculate_fraud_risk(evidence):

    score = 0

    if evidence["verification_score"] is not None:

        score += (1 - evidence["verification_score"]) * 50

    score += evidence["template_match_count"] * 20

    score += evidence["ring_match_count"] * 15

    return round(score, 2)

In [10]:
evidence = collect_claim_evidence("CLM001")

calculate_fraud_risk(evidence)

90.89

In [11]:
def build_investigation_prompt(evidence):

    return f"""
You are an Insurance Fraud Investigator.

Claim ID:
{evidence['claim_id']}

Claimant Statement:
{evidence['claimant_statement']}

Police Report:
{evidence['police_report']}

Adjuster Notes:
{evidence['adjuster_notes']}

Verification Score:
{evidence['verification_score']}

Template Fraud Matches:
{evidence['template_match_count']}

Collusion Ring Matches:
{evidence['ring_match_count']}

Provide:

1. Fraud Risk Assessment
2. Key Contradictions
3. Supporting Evidence
4. Recommended Investigation Action
"""

In [12]:
def generate_investigation_brief(claim_id):

    evidence = collect_claim_evidence(claim_id)

    prompt = build_investigation_prompt(evidence)

    response = llm(prompt, max_new_tokens=250)

    fraud_score = calculate_fraud_risk(evidence)

    return {
        "claim_id": claim_id,
        "fraud_risk_score": fraud_score,
        "report": response[0]["generated_text"],
    }

In [13]:
report = generate_investigation_brief("CLM001")

In [14]:
report

{'claim_id': 'CLM001',
 'fraud_risk_score': 90.89,
 'report': '\nYou are an Insurance Fraud Investigator.\n\nClaim ID:\nCLM001\n\nClaimant Statement:\nA vehicle rear-ended me while I was waiting at a traffic signal.\n\nPolice Report:\nWitnesses confirmed another driver caused the accident.\n\nAdjuster Notes:\nSection international though many movement.\n\nVerification Score:\n0.1821472668914981\n\nTemplate Fraud Matches:\n1\n\nCollusion Ring Matches:\n2\n\nProvide:\n\n1. Fraud Risk Assessment\n2. Key Contradictions\n3. Supporting Evidence\n4. Recommended Investigation Action\n'}

In [15]:
investigation_reports = []

In [16]:
for claim_id in claims_df["claim_id"]:

    report = generate_investigation_brief(claim_id)

    investigation_reports.append(report)

In [17]:
reports_df = pd.DataFrame(investigation_reports)

In [18]:
reports_df.head()

,claim_id,fraud_risk_score,report
0,CLM001,90.89,\nYou are an Insurance Fraud Investigator.\n\n...
1,CLM002,67.44,\nYou are an Insurance Fraud Investigator.\n\n...
2,CLM003,74.40,\nYou are an Insurance Fraud Investigator.\n\n...
3,CLM004,67.44,\nYou are an Insurance Fraud Investigator.\n\n...
4,CLM005,60.00,\nYou are an Insurance Fraud Investigator.\n\n...


In [19]:
reports_df.sort_values(by="fraud_risk_score", ascending=False).head(10)

,claim_id,fraud_risk_score,report
6,CLM007,159.76,\nYou are an Insurance Fraud Investigator.\n\n...
8,CLM009,109.71,\nYou are an Insurance Fraud Investigator.\n\n...
9,CLM010,105.00,\nYou are an Insurance Fraud Investigator.\n\n...
0,CLM001,90.89,\nYou are an Insurance Fraud Investigator.\n\n...
7,CLM008,89.33,\nYou are an Insurance Fraud Investigator.\n\n...
2,CLM003,74.40,\nYou are an Insurance Fraud Investigator.\n\n...
13,CLM014,70.00,\nYou are an Insurance Fraud Investigator.\n\n...
12,CLM013,69.36,\nYou are an Insurance Fraud Investigator.\n\n...
1,CLM002,67.44,\nYou are an Insurance Fraud Investigator.\n\n...
3,CLM004,67.44,\nYou are an Insurance Fraud Investigator.\n\n...


In [20]:
reports_df.to_csv("../data/investigation_reports.csv", index=False)

In [21]:
def generate_json_report(claim_id):

    evidence = collect_claim_evidence(claim_id)

    return {
        "claim_id": claim_id,
        "fraud_risk_score": calculate_fraud_risk(evidence),
        "verification_score": evidence["verification_score"],
        "template_match_count": evidence["template_match_count"],
        "ring_match_count": evidence["ring_match_count"],
        "recommended_action": "Manual Investigation",
    }

In [22]:
generate_json_report("CLM001")

{'claim_id': 'CLM001',
 'fraud_risk_score': 90.89,
 'verification_score': 0.1821472668914981,
 'template_match_count': 1,
 'ring_match_count': 2,
 'recommended_action': 'Manual Investigation'}

In [23]:
json_reports = []

for claim_id in claims_df["claim_id"]:

    json_reports.append(generate_json_report(claim_id))

In [24]:
json_df = pd.DataFrame(json_reports)

In [25]:
json_df.to_csv("../data/fraud_summary_reports.csv", index=False)